<a href="https://colab.research.google.com/github/xwang335/Campbell-A/blob/main/XGB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip -q install xgboost pyarrow

In [4]:
import gc
import time
import numpy as np
import pandas as pd
import xgboost as xgb

p_path = "/content/drive/My Drive/Campbell A data/preprocess_data.parquet"

df = pd.read_parquet(p_path)
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(['DATE', 'permno']).reset_index(drop=True)

sic_df = pd.get_dummies(df['sic2'], prefix='sic', dtype=np.uint8)
sic_cols_list = sic_df.columns.tolist()
df = pd.concat([df, sic_df], axis=1)

macro    = ['tbl', 'd/p', 'e/p', 'b/m', 'tms', 'dfy', 'ntis', 'svar']
features = list(df.columns)[2:96]

print(df.shape)

def generate_920_features(df, char_cols, macro_cols, sic_cols):
    X_char  = df[char_cols].to_numpy(dtype=np.float32, copy=False)
    X_macro = df[macro_cols].to_numpy(dtype=np.float32, copy=False)
    X_sic   = df[sic_cols].to_numpy(dtype=np.float32, copy=False)

    X_inter = (X_char[:, :, None] * X_macro[:, None, :]).reshape(len(df), -1)
    X_920   = np.hstack([X_char, X_inter, X_sic]).astype(np.float32, copy=False)
    return X_920

t0 = time.time()

X_all = generate_920_features(df, features, macro, sic_cols_list)
y_all = df['exret_lead1'].to_numpy(dtype=np.float32, copy=False)

meta = df[['DATE', 'permno', 'exret_lead1', 'mvel1']].copy()
dates = meta['DATE'].to_numpy()

print("X_all shape:", X_all.shape)
print("feature build time:", round(time.time() - t0, 1), "sec")

(3712808, 183)
X_all shape: (3712808, 920)
feature build time: 11.4 sec


In [5]:
year_to_idx = {}
date_years = pd.DatetimeIndex(dates).year

for year in range(1957, 2017):
    year_to_idx[year] = np.where(date_years == year)[0]

def rows_for_years(start_year, end_year):
    parts = [year_to_idx[y] for y in range(start_year, end_year + 1) if len(year_to_idx[y]) > 0]
    if len(parts) == 0:
        return np.array([], dtype=int)
    return np.concatenate(parts)

def calc_oos_r2(y, yhat):
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    denom = np.sum(y ** 2)
    if denom == 0:
        return np.nan
    return 1 - np.sum((y - yhat) ** 2) / denom

In [6]:
def build_xgb_model(n_estimators, max_depth, learning_rate, use_gpu=True):
    params = {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'objective': 'reg:squarederror',
        'random_state': 0,
        'verbosity': 0,
        'tree_method': 'hist'
    }

    if use_gpu:
        params['device'] = 'cuda'

    return xgb.XGBRegressor(**params)


# early stopping
def fit_xgb_early_stop(
    X_train, y_train, X_val, y_val,
    max_depth, learning_rate,
    max_estimators=500,
    early_stopping_rounds=20,
    use_gpu=True
):
    model = build_xgb_model(
        n_estimators=max_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        use_gpu=use_gpu
    )

    try:
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
    except Exception:
        # fallback if GPU/device config fails
        model = build_xgb_model(
            n_estimators=max_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            use_gpu=False
        )
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

    # manual early stopping via validation path if needed
    evals_result = model.evals_result()
    val_rmse = evals_result['validation_0']['rmse']

    best_iter = 0
    best_rmse = val_rmse[0]
    patience = 0

    for i in range(1, len(val_rmse)):
        if val_rmse[i] < best_rmse:
            best_rmse = val_rmse[i]
            best_iter = i
            patience = 0
        else:
            patience += 1
            if patience >= early_stopping_rounds:
                break

    best_n = best_iter + 1

    yhat_val = model.predict(X_val, iteration_range=(0, best_n))
    val_r2 = calc_oos_r2(y_val, yhat_val)

    return model, best_n, val_r2

In [7]:
# Tune depth and learning rate
def tune_xgb(
    X_train, y_train, X_val, y_val,
    depth_grid, lr_grid,
    max_estimators=500,
    early_stopping_rounds=20,
    prev_best=None,
    use_gpu=True
):
    if prev_best is None:
        depth_candidates = depth_grid
        lr_candidates = lr_grid
    else:
        prev_d, prev_lr, _ = prev_best

        depth_candidates = [d for d in depth_grid if abs(d - prev_d) <= 1]
        if len(depth_candidates) == 0:
            depth_candidates = depth_grid

        lr_candidates = [lr for lr in lr_grid if lr in [prev_lr / 10, prev_lr, prev_lr * 10]]
        lr_candidates = sorted(set([lr for lr in lr_candidates if lr in lr_grid]))
        if len(lr_candidates) == 0:
            lr_candidates = lr_grid

    best_r2 = -np.inf
    best_params = None

    for depth in depth_candidates:
        for lr in lr_candidates:
            t0 = time.time()

            model, best_n, val_r2 = fit_xgb_early_stop(
                X_train, y_train, X_val, y_val,
                max_depth=depth,
                learning_rate=lr,
                max_estimators=max_estimators,
                early_stopping_rounds=early_stopping_rounds,
                use_gpu=use_gpu
            )

            print(
                f"depth={depth}, lr={lr}, best_n={best_n}, "
                f"val_R2={val_r2:.6f}, time={time.time()-t0:.1f}s"
            )

            if val_r2 > best_r2:
                best_r2 = val_r2
                best_params = (depth, lr, best_n)

            del model
            gc.collect()

    return best_params, best_r2

In [8]:
def fit_xgb_final(X_trainval, y_trainval, X_test, depth, lr, n_estimators, use_gpu=True):
    model = build_xgb_model(
        n_estimators=n_estimators,
        max_depth=depth,
        learning_rate=lr,
        use_gpu=use_gpu
    )

    try:
        model.fit(X_trainval, y_trainval, verbose=False)
    except Exception:
        model = build_xgb_model(
            n_estimators=n_estimators,
            max_depth=depth,
            learning_rate=lr,
            use_gpu=False
        )
        model.fit(X_trainval, y_trainval, verbose=False)

    yhat_test = model.predict(X_test)
    return yhat_test

In [9]:
start_test_year = 1987
end_test_year   = 2016

DEPTH_GRID = [1, 2, 3]
LR_GRID    = [0.03, 0.05, 0.1]

MAX_ESTIMATORS = 500
EARLY_STOPPING_ROUNDS = 20
USE_GPU = True

In [10]:
all_preds = []
best_params_prev = None

for year in range(start_test_year, end_test_year + 1):
    print(f"\n--- year {year} ---")
    t_year = time.time()

    train_idx = rows_for_years(1957, year - 13)
    val_idx   = rows_for_years(year - 12, year - 1)
    test_idx  = rows_for_years(year, year)

    X_train = X_all[train_idx]
    y_train = y_all[train_idx]

    X_val = X_all[val_idx]
    y_val = y_all[val_idx]

    X_test = X_all[test_idx]

    print(f"train={len(train_idx):,}, val={len(val_idx):,}, test={len(test_idx):,}")

    best_params, best_val_r2 = tune_xgb(
        X_train, y_train, X_val, y_val,
        depth_grid=DEPTH_GRID,
        lr_grid=LR_GRID,
        max_estimators=MAX_ESTIMATORS,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        prev_best=best_params_prev,
        use_gpu=USE_GPU
    )

    best_depth, best_lr, best_n = best_params
    print(f"selected depth={best_depth}, lr={best_lr}, n_estimators={best_n}, val_R2={best_val_r2:.6f}")

    X_trainval = np.vstack([X_train, X_val]).astype(np.float32, copy=False)
    y_trainval = np.concatenate([y_train, y_val]).astype(np.float32, copy=False)

    y_test_hat = fit_xgb_final(
        X_trainval, y_trainval, X_test,
        depth=best_depth,
        lr=best_lr,
        n_estimators=best_n,
        use_gpu=USE_GPU
    )

    res = meta.iloc[test_idx].copy().reset_index(drop=True)
    res['y_pred'] = y_test_hat
    all_preds.append(res)

    best_params_prev = best_params

    del X_train, y_train, X_val, y_val, X_test
    del X_trainval, y_trainval, y_test_hat, res
    gc.collect()

    print("year time:", round(time.time() - t_year, 1), "sec")


--- year 1987 ---
train=472,278, val=764,497, test=82,404
depth=1, lr=0.03, best_n=3, val_R2=0.002149, time=24.3s
depth=1, lr=0.05, best_n=2, val_R2=0.001980, time=23.6s
depth=1, lr=0.1, best_n=1, val_R2=0.001924, time=23.7s
depth=2, lr=0.03, best_n=1, val_R2=0.000242, time=24.2s
depth=2, lr=0.05, best_n=1, val_R2=0.000015, time=24.2s
depth=2, lr=0.1, best_n=1, val_R2=-0.002487, time=23.7s
depth=3, lr=0.03, best_n=1, val_R2=0.000164, time=24.5s
depth=3, lr=0.05, best_n=1, val_R2=-0.000171, time=24.8s
depth=3, lr=0.1, best_n=1, val_R2=-0.003139, time=24.6s
selected depth=1, lr=0.03, n_estimators=3, val_R2=0.002149
year time: 247.3 sec

--- year 1988 ---
train=530,435, val=788,744, test=83,415
depth=1, lr=0.03, best_n=2, val_R2=0.003037, time=25.4s
depth=2, lr=0.03, best_n=2, val_R2=0.002717, time=25.8s
selected depth=1, lr=0.03, n_estimators=2, val_R2=0.003037
year time: 82.1 sec

--- year 1989 ---
train=588,534, val=814,060, test=81,216
depth=1, lr=0.03, best_n=2, val_R2=0.002948, tim

In [11]:
results_xgb = pd.concat(all_preds, ignore_index=True)

r2_all = calc_oos_r2(results_xgb['exret_lead1'].values, results_xgb['y_pred'].values)
print("Final OOS R^2:", round(r2_all, 6))

results_xgb.head()

save_path = "/content/drive/MyDrive/_xgb_predictions.parquet"
results_xgb.to_parquet(save_path, index=False)
print(f"saved to {save_path}")

Final OOS R^2: -9.8e-05
saved to /content/drive/MyDrive/_xgb_predictions.parquet
